In [1]:
import pandas as pd
import numpy as np
import datetime
import re
import warnings
warnings.filterwarnings('ignore')

In [2]:
end_date = pd.to_datetime('2025-12-31 23:59:59')
start_date = pd.to_datetime('2025-01-01')
today = datetime.datetime.today().year*10000 + datetime.datetime.today().month*100 + datetime.datetime.today().day

print(f'今日是：{today}')
print(f'start_date：{start_date}')
print(f'end_date：{end_date}')

今日是：20260306
start_date：2025-01-01 00:00:00
end_date：2025-12-31 23:59:59


In [3]:
# 同件不同价_年降数据分析
## 基础数据爬起过程：
### 1.读取全部输机记录,进行清洗
### 2.查找有效期在2025.1.1——2025.12.31之间的记录的零件名称唯一值(或者读取"名称.xlsx"后再进行清洗)
### 3.查找2024.12.31和2025.12.31两天的价格(11/30不会被爬取,如12/31日存在多个有效价格仅选取最后一次输机数据)
### 4.同一零件号如有多家供应商，会一同获取
### 5."2025年价格变动幅度" = ("2025年末价格" - "2024年末价格"）/"2024年末价格",如2024年末没有价格,则显示为空

## 数据分析算法与逻辑：同一供应商,上一年末价格一致,下一年末不一致判断逻辑
### 基于以上基础数据，先找到2024年末价格一致的全部零件号，再比较他们2025年末如价格(如有)不一致则"2024年末一致,2025年末不一致“为"NG";


In [3]:
#### 读取输机数据
mass =  pd.read_excel('./eps3_mass.xlsx')
mass = mass[mass['状态'].isin(['已发布','已确认','已审批'])]

In [4]:
### 数据清洗：时间数据、零件号、供应商编码
mass['生效日期'] = pd.to_datetime(mass['生效日期'].str[:10])
mass['失效日期'] = pd.to_datetime(mass['失效日期'].str[:10])
mass['创建时间'] = pd.to_datetime(mass['创建时间'].str[:10])
mass['审批通过时间'] = mass['审批通过时间'].astype(str)
mass['审批通过时间'] = pd.to_datetime(mass['审批通过时间'].str[:10])
mass['零件号'] = mass['零件号'].astype(str)
mass['供应商编码'] = mass['供应商编码'].astype(str)

In [5]:
#### 读取零件名称,零件名称数据清洗:
## 1.空格需全部去除，读取前处理；
## 2.零件名称+EPS3导出结果的零件名称"（"替换成"(","）"替换成")",读取前处理；
## 3.”\u200b“零宽度空格需全部去除，读取后处理

# df_part_names = pd.read_excel('名称.xlsx',sheet_name='Sheet3')
# part_names = df_part_names['零件名称'].tolist()
# part_names = [s.replace('\u200b', '') for s in part_names]
# len(part_names)

## 还有一种办法：读取25年全部输机记录，找到唯一零件名称
df_mass_2025 = mass[(mass['生效日期'] >= pd.to_datetime('2025-01-01')) & (mass['生效日期'] <= pd.to_datetime('2025-12-31'))]
part_names = df_mass_2025['零件名称'].unique()

### 实现功能：2024年末价格,2025年末价格,2025年价格变动幅度,三个新增字段的数据

In [ ]:
columns=['零件号','零件名称','供应商代码','供应商名称','2024年末裸价','2025年末裸价','2025年价格变动幅度'] # 7列（不加”序号“及”采购工程师“）

df_part_all_list = []

for part_name in part_names:
    
    df_part = pd.DataFrame(columns=columns)

    # 基础数据爬取整理
    
    mass_part = mass[mass['零件名称'] == part_name]                       # 完整匹配零件名称
    # mass_part = mass[mass['零件名称'].str.contains(part_name,na=False)] # 模拟匹配零件名称

    part_nos = mass_part['零件号'].unique()

    for no in part_nos:
        mass_part_no = mass_part[mass_part['零件号'] == no]
        mass_part_no_suppliers = mass_part_no['供应商编码'].unique()
        for supplier in mass_part_no_suppliers:
            mass_part_no_supplier = mass_part_no[mass_part_no['供应商编码'] == supplier]

            try:
                mass_part_no_supplier_2024 = mass_part_no_supplier[mass_part_no_supplier['失效日期'] == pd.to_datetime('2024-12-31')]
                mass_part_supplier_2024_latest = mass_part_no_supplier_2024.sort_values(by=['创建时间'],ascending=False).head(1)
                price_2024 = mass_part_supplier_2024_latest.iloc[0,7]
            except:
                price_2024 = np.nan

            try:
                mass_part_no_supplier_2025 = mass_part_no_supplier[mass_part_no_supplier['失效日期'] == pd.to_datetime('2025-12-31')]
                mass_part_supplier_2025_latest = mass_part_no_supplier_2025.sort_values(by=['创建时间'],ascending=False).head(1)
                price_2025 = mass_part_supplier_2025_latest.iloc[0,7]
            except:
                price_2025 = np.nan


            df_part.loc[df_part.shape[0]] = [no ,mass_part_no_supplier.iloc[0,4] ,mass_part_no_supplier.iloc[0,1] , mass_part_no_supplier.iloc[0,2] ,
                                            price_2024 , price_2025,
                                            f"{(price_2025-price_2024)/price_2024*100:.2f}%" 
                                            ]
            
    df_part = df_part.replace('nan%', np.nan)
    df_part.sort_values(by='零件号',inplace=True)
    df_part.reset_index(drop=True,inplace=True)
    df_part.index = df_part.index + 1
    df_part.index.name = "序号"


    ##  数据分析算法与逻辑：上一年末价格一致，下一年末价格不一致
    df_part['2024年末同一零件名称同一供应商价格一致，2025年末不一致'] = np.nan

    # ### 2022年末价格一致，2023、2024年末不一致
    # part_2024_prices = df_part.dropna(subset=['2024年末裸价'])['2024年末裸价'].unique()
    # for part_2024_price in part_2024_prices:
    #     df_part_2024_price_no = df_part[df_part['2024年末裸价'] == part_2024_price]
    
    #     for part_2024_price_no_supplier in df_part_2022_price_no['供应商代码'].unique():
        
    #         df_part_2022_price_no_supplier = df_part_2022_price_no[df_part_2022_price_no['供应商代码'] == part_2022_price_no_supplier]
    
    #         if not df_part_2022_price_no_supplier['2023年末裸价'].isna().all():
            
    #             if df_part_2022_price_no_supplier.dropna(subset=['2023年末裸价'])['2023年末裸价'].unique().size != 1:
    #                 index_need_change_price = df_part_2022_price_no_supplier.dropna(subset=['2023年末裸价'])['2023年末裸价'].index.to_list()
    #                 df_part.loc[index_need_change_price,f"2022年末同一零件名称同一供应商价格一致，2023年末不一致"] = 'NG'
    
    #         if not df_part_2022_price_no_supplier['2024年末裸价'].isna().all():
            
    #             if df_part_2022_price_no_supplier.dropna(subset=['2024年末裸价'])['2024年末裸价'].unique().size != 1:
    #                 index_need_change_price = df_part_2022_price_no_supplier.dropna(subset=['2024年末裸价'])['2024年末裸价'].index.to_list()
    #                 df_part.loc[index_need_change_price,f"2022年末同一零件名称同一供应商价格一致，2024年末不一致"] = 'NG'


    ### 2024年末价格一致，2025年末不一致
    part_2024_prices = df_part.dropna(subset=['2024年末裸价'])['2024年末裸价'].unique()

    for part_2024_price in part_2024_prices:
        df_part_2024_price_no = df_part[df_part['2024年末裸价'] == part_2024_price]

        for part_2024_price_no_supplier in df_part_2024_price_no['供应商代码'].unique():

            df_part_2024_price_no_supplier = df_part_2024_price_no[df_part_2024_price_no['供应商代码'] == part_2024_price_no_supplier]

            if not df_part_2024_price_no_supplier['2025年末裸价'].isna().all():
            
                if df_part_2024_price_no_supplier.dropna(subset=['2025年末裸价'])['2025年末裸价'].unique().size != 1:
                    index_need_change_price = df_part_2024_price_no_supplier.dropna(subset=['2025年末裸价'])['2025年末裸价'].index.to_list()
                    df_part.loc[index_need_change_price,f"2024年末同一零件名称同一供应商价格一致，2025年末不一致"] = 'NG'


    df_part_all_list.append(df_part)

df_part_all = pd.concat(df_part_all_list, ignore_index=True)
df_part_all.sort_values(by=['零件名称','供应商代码','2024年末裸价'],inplace=True)

In [ ]:
df_part_all.to_excel(f'同件不同价_年降{today}.xlsx',index=False)

In [7]:
# 排查数量
df_mass_2025['零件号'].unique().size

31035